# Interactive symmetric slab waveguide

This notebook accompanies the **Photonic Devices 2026** course.

It solves the guided modes of a symmetric dielectric slab waveguide and lets you explore interactively how the modal spectrum changes with:

- core refractive index $n_1$;
- cladding refractive index $n_2$;
- slab thickness $d$;
- wavelength $\lambda$;
- TE or TM polarization.

The notebook displays the allowed effective indices, the corresponding field profiles, and the cutoff structure.


## 1. Model

We consider a symmetric slab with propagation along $z$ and transverse coordinate $x$:

$$n(x)=\begin{cases}
n_1, & |x|<d/2,\\
n_2, & |x|>d/2,
\end{cases}\qquad n_1>n_2.$$

For a guided mode,

$$n_2k_0<\beta<n_1k_0,$$

and

$$n_2<n_{\rm eff}=\frac{\beta}{k_0}<n_1.$$

Using $a=d/2$, define

$$u=h a,\qquad w=q a,$$

with

$$h=\sqrt{n_1^2k_0^2-\beta^2},\qquad
q=\sqrt{\beta^2-n_2^2k_0^2}.$$

The normalized frequency is

$$V=k_0a\sqrt{n_1^2-n_2^2},\qquad V^2=u^2+w^2.$$


## 2. Modal equations

For TE modes, using $E_y$ as the scalar modal field,

$$u\tan u=w \qquad \text{(even)},$$

$$-u\cot u=w \qquad \text{(odd)}.$$

For TM modes, using $H_y$ as the scalar modal field,

$$u\tan u=\frac{n_1^2}{n_2^2}w \qquad \text{(even)},$$

$$-u\cot u=\frac{n_1^2}{n_2^2}w \qquad \text{(odd)}.$$

Only discrete roots of these equations correspond to guided modes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
import ipywidgets as widgets
from IPython.display import display

# Enable widgets when running in Google Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass


## 3. Numerical solver

The following cell defines the mode solver and the field-profile reconstruction. You normally do not need to modify this cell.


In [ ]:
def solve_symmetric_slab(n1, n2, d_um, wavelength_um, polarization='TE'):
    if n1 <= n2:
        return None, []

    d = d_um * 1e-6
    wavelength = wavelength_um * 1e-6
    a = d / 2
    k0 = 2 * np.pi / wavelength
    V = k0 * a * np.sqrt(n1**2 - n2**2)

    # For TM modes the boundary condition contains epsilon_1/epsilon_2.
    rho = 1.0 if polarization == 'TE' else (n1 / n2)**2

    modes = []
    eps = 1e-9

    # Mode m occupies the interval m*pi/2 < u < (m+1)*pi/2.
    # Even m -> even field; odd m -> odd field.
    for m in range(100):
        u_left = m * np.pi / 2 + eps
        u_right = min((m + 1) * np.pi / 2 - eps, V - eps)

        if u_right <= u_left:
            break

        def w_of_u(u):
            return np.sqrt(max(V**2 - u**2, 0.0))

        if m % 2 == 0:
            # even mode
            def f(u):
                return u * np.tan(u) - rho * w_of_u(u)
            parity = 'even'
        else:
            # odd mode
            def f(u):
                return -u / np.tan(u) - rho * w_of_u(u)
            parity = 'odd'

        f_left = f(u_left)
        f_right = f(u_right)

        if np.isfinite(f_left) and np.isfinite(f_right) and f_left * f_right <= 0:
            u = brentq(f, u_left, u_right, maxiter=200)
            w = np.sqrt(V**2 - u**2)
            h = u / a
            q = w / a
            beta = np.sqrt((n1 * k0)**2 - h**2)
            neff = beta / k0

            modes.append({
                'm': m,
                'parity': parity,
                'u': u,
                'w': w,
                'h': h,
                'q': q,
                'beta': beta,
                'neff': neff
            })

    return V, modes


def scalar_field_profile(x, mode, d_um):
    d = d_um * 1e-6
    a = d / 2
    h = mode['h']
    q = mode['q']
    m = mode['m']

    field = np.zeros_like(x)
    core = np.abs(x) <= a
    right = x > a
    left = x < -a

    if m % 2 == 0:
        # even mode
        field[core] = np.cos(h * x[core])
        boundary_value = np.cos(h * a)
        field[right] = boundary_value * np.exp(-q * (x[right] - a))
        field[left] = boundary_value * np.exp(-q * (-x[left] - a))
    else:
        # odd mode
        field[core] = np.sin(h * x[core])
        boundary_value = np.sin(h * a)
        field[right] = boundary_value * np.exp(-q * (x[right] - a))
        field[left] = -boundary_value * np.exp(-q * (-x[left] - a))

    max_field = np.max(np.abs(field))
    if max_field > 0:
        field = field / max_field

    return field


## 4. Interactive slab explorer

Move the sliders and observe how the guided-mode spectrum changes. In particular, watch modes appear or disappear when a cutoff is crossed.


In [ ]:
def update_slab(n1, n2, d_um, wavelength_um, polarization):
    if n1 <= n2:
        print('Guiding condition not satisfied: choose n1 > n2.')
        return

    V, modes = solve_symmetric_slab(n1, n2, d_um, wavelength_um, polarization)

    print(f'Polarization: {polarization}')
    print(f'Normalized frequency V = {V:.3f}')
    print(f'Number of guided modes = {len(modes)}')

    if not modes:
        print('No guided solutions found.')
        return

    # ------------------------------------------------------------
    # Modal table
    # ------------------------------------------------------------
    rows = []
    for mode in modes:
        rows.append({
            'mode': f"{polarization}{mode['m']}",
            'parity': mode['parity'],
            'n_eff': mode['neff'],
            'beta (1/um)': mode['beta'] * 1e-6,
            'q (1/um)': mode['q'] * 1e-6
        })

    table = pd.DataFrame(rows)
    display(table.style.format({
        'n_eff': '{:.6f}',
        'beta (1/um)': '{:.4f}',
        'q (1/um)': '{:.4f}'
    }))

    # ------------------------------------------------------------
    # 1. Discrete effective-index spectrum
    # ------------------------------------------------------------
    plt.figure(figsize=(8, 2.3))
    plt.plot([n2, n1], [0, 0], linewidth=1.5)
    neffs = [mode['neff'] for mode in modes]
    plt.scatter(neffs, np.zeros(len(neffs)), s=55, zorder=3)

    for mode in modes:
        plt.text(mode['neff'], 0.035, f"{polarization}{mode['m']}",
                 ha='center', va='bottom')

    plt.axvline(n2, linestyle=':')
    plt.axvline(n1, linestyle=':')
    plt.xlim(max(0, n2 - 0.08*(n1-n2)), n1 + 0.08*(n1-n2))
    plt.ylim(-0.12, 0.16)
    plt.yticks([])
    plt.xlabel(r'Effective index $n_{\rm eff}$')
    plt.title('Discrete guided-mode spectrum')
    plt.grid(True, axis='x')
    plt.show()

    # ------------------------------------------------------------
    # 2. Scalar modal field profiles
    # ------------------------------------------------------------
    d = d_um * 1e-6
    x_max = 2.5 * d
    x = np.linspace(-x_max, x_max, 1600)

    plt.figure(figsize=(8, 4.6))
    for mode in modes:
        field = scalar_field_profile(x, mode, d_um)
        label = f"{polarization}{mode['m']}   n_eff={mode['neff']:.4f}"
        plt.plot(x * 1e6, field, label=label)

    plt.axvline(-d_um/2, linestyle=':')
    plt.axvline(+d_um/2, linestyle=':')
    plt.xlabel(r'Transverse coordinate $x$ ($\mu$m)')
    ylabel = r'Normalized $E_y$' if polarization == 'TE' else r'Normalized $H_y$'
    plt.ylabel(ylabel)
    plt.title('Guided-mode field profiles')
    plt.grid(True)
    plt.legend()
    plt.show()

    # ------------------------------------------------------------
    # 3. Cutoff ladder
    # ------------------------------------------------------------
    vmax = max(V * 1.25, np.pi)
    max_m = int(np.floor(2 * vmax / np.pi)) + 1

    plt.figure(figsize=(8, 2.2))
    plt.plot([0, vmax], [0, 0], linewidth=1.5)

    # Fundamental mode has no cutoff; higher-order modes appear at V=m*pi/2.
    for m in range(1, max_m + 1):
        Vc = m * np.pi / 2
        if Vc <= vmax:
            plt.axvline(Vc, linestyle=':')
            plt.text(Vc, 0.03, f"{polarization}{m} cutoff",
                     rotation=90, ha='center', va='bottom')

    plt.scatter([V], [0], s=65, zorder=3)
    plt.text(V, -0.035, f'current V = {V:.2f}', ha='center', va='top')
    plt.xlim(0, vmax)
    plt.ylim(-0.13, 0.18)
    plt.yticks([])
    plt.xlabel('Normalized frequency V')
    plt.title('Modal cutoffs')
    plt.grid(True, axis='x')
    plt.show()


n1_slider = widgets.FloatSlider(
    value=1.50, min=1.05, max=4.00, step=0.05,
    description='n1:', continuous_update=False,
    style={'description_width': '40px'}, layout=widgets.Layout(width='350px'))

n2_slider = widgets.FloatSlider(
    value=1.00, min=1.00, max=3.00, step=0.05,
    description='n2:', continuous_update=False,
    style={'description_width': '40px'}, layout=widgets.Layout(width='350px'))

d_slider = widgets.FloatSlider(
    value=0.80, min=0.10, max=3.00, step=0.05,
    description='d (um):', continuous_update=False,
    style={'description_width': '70px'}, layout=widgets.Layout(width='350px'))

lambda_slider = widgets.FloatSlider(
    value=1.55, min=0.40, max=2.50, step=0.05,
    description='lambda (um):', continuous_update=False,
    style={'description_width': '90px'}, layout=widgets.Layout(width='350px'))

pol_selector = widgets.ToggleButtons(
    options=['TE', 'TM'], value='TE', description='Polarization:')

controls = widgets.VBox([
    widgets.HBox([n1_slider, n2_slider]),
    widgets.HBox([d_slider, lambda_slider]),
    pol_selector
])

interactive_output = widgets.interactive_output(
    update_slab,
    {
        'n1': n1_slider,
        'n2': n2_slider,
        'd_um': d_slider,
        'wavelength_um': lambda_slider,
        'polarization': pol_selector
    }
)

display(controls, interactive_output)


## 5. Suggested exploration

Before moving a slider, try to predict what will happen.

1. **Increase the slab thickness $d$.** How does the number of guided modes change?
2. **Increase the wavelength $\lambda$.** Does confinement become stronger or weaker?
3. **Reduce the index contrast $n_1-n_2$.** What happens to $n_{\rm eff}$ and to the evanescent tails?
4. Switch between **TE and TM**. Why are their effective indices different?
5. Choose parameters close to a modal cutoff. What happens to $n_{\rm eff}$ and to the field extension in the cladding?

A useful observation is that a mode approaching cutoff satisfies

$$n_{\rm eff}\rightarrow n_2,$$

and its evanescent decay constant $q$ tends to zero.
